In [1]:
# Install
!pip install pandas scikit-learn plotly

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ Loading ND-GAIN data...")

# Full paths after extraction
vuln_path = "Resources/resources/vulnerability/vulnerability.csv"
gdp_path = "Resources/resources/indicators/gdp/score.csv"
pop_path = "Resources/resources/indicators/pop/score.csv"

# Load
vulnerability = pd.read_csv(vuln_path)
gdp_data = pd.read_csv(gdp_path)
pop_data = pd.read_csv(pop_path)

print(f"✅ Vulnerability shape: {vulnerability.shape}")
print(f"✅ GDP shape: {gdp_data.shape}")
print(f"✅ Population shape: {pop_data.shape}")

✅ Loading ND-GAIN data...


FileNotFoundError: [Errno 2] No such file or directory: 'Resources/resources/vulnerability/vulnerability.csv'

In [ ]:
# Load modern emissions data
print("🌍 Loading modern CO2 data from Our World in Data...")
owid = pd.read_csv("https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv")

# Get latest year per country (2022 or 2023)
owid_latest = owid.sort_values('year').groupby('country').last().reset_index()
owid_clean = owid_latest[['country', 'co2']].rename(columns={'co2': 'co2_kt'})

# Compute cumulative CO2 (1990–2023)
co2_recent = owid[(owid['year'] >= 1990) & (owid['co2'].notna())]
cumulative = co2_recent.groupby('country')['co2'].sum().reset_index()
cumulative = cumulative.rename(columns={'co2': 'cumulative_co2'})

# Merge with ND-GAIN
full_data = ndgain.merge(cumulative, on='country', how='inner')
print(f"✅ Final dataset: {len(full_data)} countries")

🌍 Loading modern CO2 data from Our World in Data...
✅ Final dataset: 171 countries


In [ ]:
# Climate Justice Score = Vulnerability / (Cumulative CO2 + 1)
full_data['justice_score'] = full_data['vulnerability'] / (full_data['cumulative_co2'] + 1)

# Normalize for visualization
full_data['justice_score_norm'] = (full_data['justice_score'] - full_data['justice_score'].min()) / (full_data['justice_score'].max() - full_data['justice_score'].min())

# Cluster into groups
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

features = ['cumulative_co2', 'vulnerability', 'gdp_score']
X = full_data[features].copy().dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42)
full_data.loc[X.index, 'group'] = kmeans.fit_predict(X_scaled)

# Label groups
group_labels = {
    0: "High Emitter, Low Vulnerability",
    1: "Low Emitter, High Vulnerability",  # ← Priority!
    2: "Balanced"
}
full_data['group_label'] = full_data['group'].map(group_labels)

# Priority countries
priority = full_data[full_data['group_label'] == "Low Emitter, High Vulnerability"]
priority = priority.sort_values('justice_score', ascending=False)

print("\n🚨 CLIMATE JUSTICE PRIORITY COUNTRIES (Need Global Support):")
print(priority[['country', 'vulnerability', 'cumulative_co2', 'justice_score']].head(10).to_string(index=False))

# Save
priority[['country', 'vulnerability', 'cumulative_co2', 'justice_score']].to_csv("climate_justice_priority.csv", index=False)
print("\n💾 Saved to: climate_justice_priority.csv")

# Plot
import plotly.express as px
fig = px.scatter(
    full_data,
    x='cumulative_co2',
    y='vulnerability',
    color='group_label',
    size='justice_score_norm',
    hover_name='country',
    title="🌍 Climate Justice Lens: Who Suffers Most from a Crisis They Didn’t Create?",
    labels={
        'cumulative_co2': 'Cumulative CO₂ Emissions (Mt, 1990–2023)',
        'vulnerability': 'Climate Vulnerability (ND-GAIN 2014)',
        'justice_score_norm': 'Justice Priority'
    },
    color_discrete_map={
        "Low Emitter, High Vulnerability": "red",
        "High Emitter, Low Vulnerability": "blue",
        "Balanced": "green"
    }
)
fig.update_layout(xaxis_type='log')
fig.show()


🚨 CLIMATE JUSTICE PRIORITY COUNTRIES (Need Global Support):
    country  vulnerability  cumulative_co2  justice_score
    Grenada       0.389718           7.679       0.044904
 Seychelles       0.454551          13.202       0.032006
Saint Lucia       0.354692          13.410       0.024614
   Barbados       0.362759          44.912       0.007901
    Bahamas       0.452721          70.267       0.006352
 Montenegro       0.370827          66.014       0.005534
   Suriname       0.404046          76.159       0.005237
      Malta       0.326775          79.691       0.004050
  Mauritius       0.437997         109.105       0.003978
    Iceland       0.324717         106.128       0.003031

💾 Saved to: climate_justice_priority.csv


In [ ]:
# Save your notebook and CSV
!jupyter nbconvert --to notebook --output "climate_justice_lens.ipynb" /content/*.ipynb

# List files to confirm
!ls -l

[NbConvertApp] WARNING | pattern '/content/*.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    